In [5]:
!uv pip install adabelief-pytorch

Using Python 3.12.12 environment at: /usr
Resolved 29 packages in 305ms                                        
Prepared 1 package in 23ms                                               
Installed 1 package in 4ms0.2.1                             
 + adabelief-pytorch==0.2.1


In [46]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import tensorflow as tf
import numpy as np
vocab_size = 20000

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

In [47]:
x_test.shape

(25000,)

In [48]:
import numpy as np

lengths = [len(seq) for seq in x_train]

max_len = int(np.percentile(lengths, 95))

print("Chosen max_len:", max_len)

Chosen max_len: 610


In [49]:
x_train = x_train
y_train = y_train

# x_test = x_test[:5000]
# y_test = y_test[:5000]

x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test = tf.keras.preprocessing.sequence.pad_sequences(x_test, maxlen=max_len)

x_train = torch.tensor(x_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)

x_test = torch.tensor(x_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)
train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

In [50]:
x_train.shape

torch.Size([25000, 610])

In [51]:
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    pin_memory=True
)

In [37]:
class SentimentRNN(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.norm = nn.BatchNorm1d(hidden_dim * 2)

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):

        x = self.embedding(x)

        output, hidden = self.rnn(x)

        # take last layer hidden state
        hidden = hidden[-2:].permute(1,0,2).reshape(x.size(0), -1)

        hidden = self.norm(hidden)

        out = self.fc(hidden)

        return out

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SentimentRNN(
    vocab_size=20000,
    embed_dim=128,
    hidden_dim=128
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

model = torch.compile(model)

In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for x, y in train_loader:

        optimizer.zero_grad(set_to_none=True)

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        train_correct += (preds == y).sum().item()
        train_total += y.size(0)


    train_acc = train_correct / train_total


    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in test_loader:

            outputs = model(x)

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)


    test_acc = correct / total


    print(
        f"Epoch {epoch+1} | "
        f"Train Acc {train_acc:.4f} | "
        f"Test Acc {test_acc:.4f}"
    )